In [1]:
from google.colab import drive
drive.mount("/content/drive")

import importlib.util, subprocess, sys
if importlib.util.find_spec("torch_geometric") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])

import ast, json, os, random, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/content/drive/MyDrive/Threat-detection/gnn_edge_data"
GRAPH_DIR = os.path.join(DATA_DIR, "prepared_graphs")
MODEL_DIR = os.path.join(DATA_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(DATA_DIR, "train_edges_5min_enriched.parquet")
VAL_FILE = os.path.join(DATA_DIR, "validation_edges_5min_enriched.parquet")
TEST_FILE = os.path.join(DATA_DIR, "test_edges_5min_enriched.parquet")
LABEL_COLUMN = "tactic_targets"

train_graphs = torch.load(os.path.join(GRAPH_DIR, "train_graphs.pt"), map_location="cpu", weights_only=False)
val_graphs = torch.load(os.path.join(GRAPH_DIR, "validation_graphs.pt"), map_location="cpu", weights_only=False)
test_graphs = torch.load(os.path.join(GRAPH_DIR, "test_graphs.pt"), map_location="cpu", weights_only=False)

def load_sorted_labels(path):
    return (
        pd.read_parquet(
            path,
            columns=["window_start", LABEL_COLUMN]
        )
        .sort_values(
            "window_start",
            kind="quicksort"
        )
        .reset_index(drop=True)
    )

train_labels = load_sorted_labels(TRAIN_FILE)
val_labels = load_sorted_labels(VAL_FILE)
test_labels = load_sorted_labels(TEST_FILE)

print("Device:", device)
print("Graphs:", len(train_graphs), len(val_graphs), len(test_graphs))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Graphs: 9300 10560 5326


In [2]:
def parse_list(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return list(value)
    if pd.isna(value):
        return []
    text = str(value).strip()
    if text.startswith("["):
        try:
            return list(ast.literal_eval(text))
        except (ValueError, SyntaxError):
            pass
    return [text]


def clean_labels(value):
    labels = parse_list(value)
    blocked = {"", "none", "unknown", "nan", "null"}
    return sorted({str(x).strip() for x in labels if str(x).strip().lower() not in blocked})


classes = sorted({label for value in train_labels[LABEL_COLUMN] for label in clean_labels(value)})
class_to_id = {label: i for i, label in enumerate(classes)}

def graph_row_ids(graph):
    if hasattr(graph, "target_row_ids"):
        return graph.target_row_ids.cpu().numpy()
    return graph.target_row_index.cpu().numpy()

def attach_targets(graphs, label_frame):
    output = []
    for graph in graphs:
        row_ids = graph_row_ids(graph)
        keep, targets = [], []
        for edge_id, row_id in enumerate(row_ids):
            labels = [x for x in clean_labels(label_frame.iloc[int(row_id)][LABEL_COLUMN]) if x in class_to_id]
            if int(graph.y[edge_id]) == 1 and labels:
                target = np.zeros(len(classes), dtype=np.float32)
                target[[class_to_id[x] for x in labels]] = 1.0
                keep.append(edge_id)
                targets.append(target)
        if keep:
            keep = torch.tensor(keep, dtype=torch.long)
            item = graph.clone()
            item.target_edge_index = graph.target_edge_index[:, keep]
            item.target_edge_attr = graph.target_edge_attr[keep]
            item.y = graph.y[keep]
            if hasattr(item, "target_row_ids"):
                item.target_row_ids = graph.target_row_ids[keep]
            if hasattr(item, "target_row_index"):
                item.target_row_index = graph.target_row_index[keep]
            item.label_y = torch.tensor(np.stack(targets), dtype=torch.float32)
            output.append(item)
    return output

train_data = attach_targets(train_graphs, train_labels)
val_data = attach_targets(val_graphs, val_labels)
test_data = attach_targets(test_graphs, test_labels)

print("Classes:", len(classes))
print("Labeled edges:", sum(len(g.label_y) for g in train_data), sum(len(g.label_y) for g in val_data), sum(len(g.label_y) for g in test_data))
print(classes)

Classes: 13
Labeled edges: 779 52913 958
['Collection', 'Command and Control', 'Credential Access', 'Defense Evasion', 'Discovery', 'Execution', 'Exfiltration', 'Initial Access', 'Lateral Movement', 'Persistence', 'Privilege Escalation', 'Reconnaissance', 'Resource Development']


In [3]:
class EdgeMultiLabelGraphSAGE(nn.Module):
    def __init__(self, num_classes, hidden_size=64, dropout=0.2):
        super().__init__()
        self.conv1 = SAGEConv(6, hidden_size)
        self.conv2 = SAGEConv(hidden_size, hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2 + 10, hidden_size),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_size, num_classes)
        )
        self.dropout = dropout

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, data.edge_index))
        src, dst = data.target_edge_index
        return self.classifier(torch.cat([x[src], x[dst], data.target_edge_attr], dim=1))

def collect(model, loader):
    model.eval()
    probabilities, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            probabilities.append(torch.sigmoid(model(batch)).cpu())
            targets.append(batch.label_y.cpu())
    return torch.cat(probabilities).numpy(), torch.cat(targets).numpy().astype(int)

def scores(targets, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "micro_pr_auc": float(average_precision_score(targets, probabilities, average="micro")),
        "micro_precision": float(precision_score(targets, predictions, average="micro", zero_division=0)),
        "micro_recall": float(recall_score(targets, predictions, average="micro", zero_division=0)),
        "micro_f1": float(f1_score(targets, predictions, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),
    }

In [4]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

model = EdgeMultiLabelGraphSAGE(len(classes)).to(device)
all_targets = torch.cat([g.label_y for g in train_data])
positive = all_targets.sum(0)
pos_weight = ((len(all_targets) - positive) / positive.clamp_min(1)).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)

best_score, waiting = -1.0, 0
checkpoint_path = os.path.join(MODEL_DIR, "tactic_graphsage.pt")

for epoch in range(1, 21):
    model.train()
    total_loss, total_edges = 0.0, 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch), batch.label_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch.label_y)
        total_edges += len(batch.label_y)

    val_prob, val_true = collect(model, val_loader)
    val_ap = average_precision_score(val_true, val_prob, average="micro")
    print(f"Epoch {epoch:02d} | Loss {total_loss / total_edges:.4f} | Val micro PR-AUC {val_ap:.4f}")

    if val_ap > best_score:
        best_score, waiting = val_ap, 0
        torch.save({"model_state_dict": model.state_dict(), "classes": classes}, checkpoint_path)
    else:
        waiting += 1
    if waiting >= 5:
        print("Early stopping")
        break

Epoch 01 | Loss 1.3485 | Val micro PR-AUC 0.0587
Epoch 02 | Loss 1.2760 | Val micro PR-AUC 0.0601
Epoch 03 | Loss 1.2361 | Val micro PR-AUC 0.0619
Epoch 04 | Loss 1.2125 | Val micro PR-AUC 0.0607
Epoch 05 | Loss 1.1657 | Val micro PR-AUC 0.0542
Epoch 06 | Loss 1.1464 | Val micro PR-AUC 0.0552
Epoch 07 | Loss 1.1092 | Val micro PR-AUC 0.0566
Epoch 08 | Loss 1.0907 | Val micro PR-AUC 0.0576
Early stopping


In [5]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

val_prob, val_true = collect(model, val_loader)
thresholds = np.arange(0.10, 0.91, 0.05)
best_threshold = max(thresholds, key=lambda t: f1_score(val_true, val_prob >= t, average="micro", zero_division=0))
validation_metrics = scores(val_true, val_prob, best_threshold)

test_prob, test_true = collect(model, test_loader)
test_metrics = scores(test_true, test_prob, best_threshold)

final_checkpoint = {
    "model_state_dict": model.state_dict(),
    "classes": classes,
    "threshold": float(best_threshold),
    "validation_metrics": validation_metrics,
    "test_metrics": test_metrics,
}
torch.save(final_checkpoint, checkpoint_path)

metrics_path = os.path.join(MODEL_DIR, "tactic_metrics.json")
with open(metrics_path, "w") as file:
    json.dump({"threshold": float(best_threshold), "validation": validation_metrics, "test": test_metrics}, file, indent=2)

print("Threshold:", round(float(best_threshold), 2))
print("Validation:", validation_metrics)
print("Test:", test_metrics)
print("Saved:", checkpoint_path)

Threshold: 0.3
Validation: {'micro_pr_auc': 0.06191326657940768, 'micro_precision': 0.08194405199208817, 'micro_recall': 0.99983495020998, 'micro_f1': 0.1514736872812946, 'macro_f1': 0.10980796830918377}
Test: {'micro_pr_auc': 0.061568695906454196, 'micro_precision': 0.08470964304741609, 'micro_recall': 0.9958246346555324, 'micro_f1': 0.15613747954173485, 'macro_f1': 0.10205527706248602}
Saved: /content/drive/MyDrive/Threat-detection/gnn_edge_data/models/tactic_graphsage.pt


In [6]:
def check_alignment(graphs, parquet_path, sort_kind):
    frame = pd.read_parquet(
        parquet_path,
        columns=["window_start", "binary_target"]
    )

    frame = (
        frame
        .sort_values(
            "window_start",
            kind=sort_kind
        )
        .reset_index(drop=True)
    )

    correct = 0
    total = 0

    for graph in graphs:
        row_ids = graph_row_ids(graph)

        expected = (
            frame.iloc[row_ids]["binary_target"]
            .to_numpy()
            .astype(int)
        )

        actual = (
            graph.y.cpu()
            .numpy()
            .astype(int)
        )

        correct += (expected == actual).sum()
        total += len(actual)

    return correct / total


for kind in ["quicksort", "stable"]:
    print(
        kind,
        "| Train alignment:",
        check_alignment(
            train_graphs,
            TRAIN_FILE,
            kind
        ),
        "| Validation alignment:",
        check_alignment(
            val_graphs,
            VAL_FILE,
            kind
        ),
        "| Test alignment:",
        check_alignment(
            test_graphs,
            TEST_FILE,
            kind
        )
    )

quicksort | Train alignment: 1.0 | Validation alignment: 1.0 | Test alignment: 1.0
stable | Train alignment: 1.0 | Validation alignment: 0.9997627536840964 | Test alignment: 0.9995088070042397
